<a href="https://colab.research.google.com/github/MaiAlhusseini/FlyRank_Intern_repo/blob/main/Copy_of_w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaiAlhusseini/FlyRank_Intern_repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Content Lifecycle: Growing vs Declining

The FlyRank research paper compares pages that are gaining traffic with pages that are losing traffic. The reported averages show that growing pages are younger than declining pages: 185 days versus 228 days. Average word count is almost identical between the two groups, at approximately 1,487 words for growing pages and 1,481 words for declining pages.

#### Methodology questions

**Where does the label/group come from?**

The groups are based on traffic trend direction: pages are classified as gaining or losing traffic according to their observed performance trend. Therefore, the group label is derived from historical traffic behavior rather than from a future business outcome such as whether a refresh was successful.

**Does the validation design support the claim?**

The comparison supports an observed association between content age and the growing/declining groups. However, this comparison alone does not establish that older content causes traffic decline. Other factors could also differ between the groups.

Therefore, the evidence supports language such as "older pages were associated with declining traffic in this dataset" rather than "older content causes traffic decline."

---

### Finding 4 — The Freshness Multiplier

The paper examines performance across different freshness windows and also compares refreshed and stale pages in older-content cohorts. The report shows substantially stronger observed performance for some recently refreshed older pages compared with stale comparison groups.

#### Methodology questions

**Where does the label/group come from?**

The comparison groups are based on page freshness and whether pages were recently refreshed versus remaining stale. These are observed historical states rather than a supervised-learning target showing that a page was randomly assigned to receive a refresh.

**Does the validation design support the claim?**

The comparisons provide evidence of an observed difference between refreshed and stale pages, and the paper reports statistical significance tests for the refreshed-versus-stale comparison. However, if pages were selected for refresh based on their existing characteristics, selection bias or confounding could contribute to the observed difference.

Therefore, the evidence is useful for decision-support and shows an association between freshness/refresh status and observed performance, but it should not automatically be interpreted as proof that refreshing any page will cause a performance improvement.

---

### Overall methodology lesson

The two findings show why the source of the grouping or label and the validation design matter.

A measured difference between groups can support an observed association, but stronger causal language requires a stronger experimental or quasi-experimental design that addresses alternative explanations and selection effects.


## 2. My model under an honest split (before/after)

In Week 5, I used a client-grouped split so that pages from the same client could not appear in both the training and testing sets.

For this validation audit, I compare two validation designs:

- A random row split, which is included as a diagnostic "before" result.
- A client-grouped split, which is treated as the more honest "after" evaluation.

The random split can place pages from the same client in both training and testing data. Because pages from the same client may share characteristics, this can produce an optimistic estimate of generalization.

The grouped split holds entire clients out of training and evaluates the model on previously unseen clients. This better matches the question of whether the model can generalize across clients.

I keep the target, model type, feature set, and Precision@50 metric consistent between the two evaluations so that the main difference is the validation design.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from google.colab import files


uploaded = files.upload()
df = pd.read_csv("content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

# Create the Week-5 target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nDeclining base rate:")
print(round(df["is_declining_label"].mean(), 3))

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
Dataset shape: (30000, 44)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining base rate:
0.542


In [ ]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

# Keep only columns that exist in the dataset
numeric_features = [
    col for col in numeric_features
    if col in df.columns
]

categorical_features = [
    col for col in categorical_features
    if col in df.columns
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

print("Number of features:", len(feature_columns))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Number of features: 22
Numeric features: 14
Categorical features: 8


In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

def create_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

In [ ]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_k_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_k_indices].mean()

In [ ]:
# -------------------------------
# BEFORE: Random row split
# -------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = create_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_p50 = precision_at_k(
    y_test_random,
    random_scores,
    k=50
)

random_base_rate = y_test_random.mean()

print("Random split results")
print("--------------------")
print("Training rows:", len(X_train_random))
print("Test rows:", len(X_test_random))
print("Base rate:", round(random_base_rate, 3))
print("Precision@50:", round(random_p50, 3))

Random split results
--------------------
Training rows: 24000
Test rows: 6000
Base rate: 0.542
Precision@50: 0.94


In [ ]:
# ---------------------------------------
# AFTER: Client-grouped split
# ---------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

grouped_model = create_model()

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_p50 = precision_at_k(
    y_test_grouped,
    grouped_scores,
    k=50
)

grouped_base_rate = y_test_grouped.mean()

client_overlap = set(groups_train).intersection(
    set(groups_test)
)

print("Client-grouped split results")
print("----------------------------")
print("Training rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())
print("Client overlap:", len(client_overlap))
print("Base rate:", round(grouped_base_rate, 3))
print("Precision@50:", round(grouped_p50, 3))

Client-grouped split results
----------------------------
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
Base rate: 0.511
Precision@50: 0.6


In [ ]:
comparison = pd.DataFrame({
    "validation_design": [
        "Random row split",
        "Client-grouped split"
    ],
    "test_rows": [
        len(y_test_random),
        len(y_test_grouped)
    ],
    "test_clients": [
        groups.iloc[y_test_random.index].nunique(),
        groups_test.nunique()
    ],
    "base_rate": [
        random_base_rate,
        grouped_base_rate
    ],
    "precision_at_50": [
        random_p50,
        grouped_p50
    ]
})

comparison["precision_minus_base_rate"] = (
    comparison["precision_at_50"]
    - comparison["base_rate"]
)

display(comparison)

,validation_design,test_rows,test_clients,base_rate,precision_at_50,precision_minus_base_rate
0,Random row split,6000,31,0.542000,0.94,0.398000
1,Client-grouped split,6163,7,0.510952,0.60,0.089048


### Before vs. after interpretation

The random row split produced a Precision@50 of 0.94, while the client-grouped split produced a Precision@50 of 0.60.

This is a difference of 0.34, or 34 percentage points. The random split therefore gives a substantially more optimistic estimate of performance than the client-grouped evaluation.

The random split allows pages from the same client to appear in both training and testing data. In contrast, the grouped split holds entire clients out of training. The grouped test set contains 7 clients with zero overlap with the 25 training clients.

I cannot conclude from this comparison alone that the model was memorizing clients. However, the large performance gap shows that validation design has a material effect on the measured performance of this model.

The grouped result is the more appropriate estimate for generalization to previously unseen clients.

The grouped test set has a declining-class base rate of 0.511, while the model's Precision@50 is 0.600. Therefore, the model's top-50 ranking is approximately 8.9 percentage points above the observed declining-class base rate in the grouped test set.

For this reason, I use the grouped Precision@50 of 0.60 as the more conservative result when describing the model's performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.